# 01 — ADME EDA (Part 1) - Biogen Dataset

**Goal**: Understand the ADME dataset thoroughly before making any cleaning or modelling decisions.  
**Dataset**: `data/raw/ADME_public_set_3521.csv` — 3521 compounds, 6 log-transformed ADME endpoints.  
**Outputs**: Findings documented in Section 1.10; no data modified here.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from src.eda import smiles_validity_report, missing_value_report, max_corr_report
from src.features import rdkit_descriptors
from src.plotting import endpoint_distributions

SEED = 42
DATA_PATH = '../data/raw/ADME_public_set_3521.csv'

TRAIN_MPNN2 = True        # set True to train MPNN2 (graph + rdkit_2d_normalized)
TRAIN_MPNN_GRAPH = True   # set True to train MPNN (graph-only, no rdkit descriptors)
TUNE_MPNN2 = True # set True to run grid search (~30-60 min)
RUN_RADIUS_SENSITIVITY = False  # set True to run 2.4b radius sweep (~2 min)

ENDPOINT_COLS = [
    'LOG HLM_CLint (mL/min/kg)',
    'LOG MDR1-MDCK ER (B-A/A-B)',
    'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'LOG PLASMA PROTEIN BINDING (HUMAN) (% unbound)',
    'LOG PLASMA PROTEIN BINDING (RAT) (% unbound)',
    'LOG RLM_CLint (mL/min/kg)',
]
EP_SHORT = ['HLM', 'MDR1', 'SOL', 'PPB_H', 'PPB_R', 'RLM']
EP_SHORT_MAP = dict(zip(ENDPOINT_COLS, EP_SHORT))

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.4f}'.format)
print('Imports OK')

## 1.1 — Load & Inspect

In [ ]:
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
print('\nColumns:', df.columns.tolist())
print('\nDtypes:')
print(df.dtypes)
df.head()

## 1.2 — SMILES Validity Report

Report only — no rows dropped here. Dropping is a cleaning decision made after EDA.

In [ ]:
validity = smiles_validity_report(df, smiles_col='SMILES')
print(f"Valid SMILES  : {validity['valid_count']} ({validity['valid_count']/len(df)*100:.2f}%)")
print(f"Invalid SMILES: {validity['invalid_count']} ({validity['invalid_count']/len(df)*100:.2f}%)")
if validity['invalid_indices']:
    print(f"Invalid row indices: {validity['invalid_indices']}")
    print(df.loc[validity['invalid_indices'], ['Internal ID', 'SMILES']])
else:
    print('No invalid SMILES found.')

## 1.3 — Duplicate Check

In [ ]:
from rdkit import Chem

# Duplicate raw SMILES strings
dup_raw = df['SMILES'].duplicated(keep=False)
print(f"Duplicate raw SMILES: {dup_raw.sum()} rows ({df['SMILES'].duplicated().sum()} extra copies)")

# Canonical SMILES duplicates (catches representation variants)
def to_canonical(smi):
    if pd.isna(smi):
        return None
    mol = Chem.MolFromSmiles(str(smi))
    return Chem.MolToSmiles(mol) if mol is not None else None

df['canonical_smiles'] = df['SMILES'].apply(to_canonical)
dup_can = df['canonical_smiles'].duplicated(keep=False) & df['canonical_smiles'].notna()
n_dup_can = df['canonical_smiles'].duplicated().sum()
print(f"Duplicate canonical SMILES: {dup_can.sum()} rows ({n_dup_can} extra copies)")

if n_dup_can > 0:
    dup_df = df[dup_can].sort_values('canonical_smiles')
    # Check endpoint consistency among duplicates
    dup_ep_std = dup_df.groupby('canonical_smiles')[ENDPOINT_COLS].std()
    inconsistent = (dup_ep_std > 0).any(axis=1)
    print(f"Duplicates with inconsistent endpoint values: {inconsistent.sum()}")
    display(dup_df[['Internal ID', 'canonical_smiles'] + ENDPOINT_COLS].head(20))

SMILES = string encoding of molecule
Canoncial SMILES = single, deterministic string generated via an algo (RDKit), always produces the same SMILES regardless of how molecule originally encoded.

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

# ── 1. Show the transformation on synthetic examples ──────────────────────────
non_canonical_examples = [
    ('Ethanol (reversed)', 'OCC'),
    ('Aspirin (variant A)',  'OC(=O)c1ccccc1OC(C)=O'),
    ('Aspirin (variant B)',  'CC(=O)Oc1ccccc1C(O)=O'),
]

rows = []
for name, smi in non_canonical_examples:
    mol = Chem.MolFromSmiles(smi)
    canon = Chem.MolToSmiles(mol)
    rows.append({'Name': name, 'Input SMILES': smi, 'Canonical SMILES': canon,
                 'Changed?': '✓' if smi != canon else '—'})

print("Synthetic examples — SMILES → Canonical:")
display(pd.DataFrame(rows))

# ── 2. Draw aspirin variants side-by-side to show same structure ──────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ax, (name, smi) in zip(axes, non_canonical_examples[1:]):
    mol = Chem.MolFromSmiles(smi)
    canon = Chem.MolToSmiles(mol)
    img = Draw.MolToImage(mol, size=(320, 220))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'Input:  {smi}\n→ Canon: {canon}', fontsize=8)
plt.suptitle('Same molecule (aspirin), two input strings → identical canonical SMILES', fontsize=10)
plt.tight_layout()
plt.show()

# ── 3. Verify our dataset: original vs canonical for first 5 compounds ─────────
print("\nDataset spot-check — first 5 compounds:")
check = df[['Internal ID', 'SMILES', 'canonical_smiles']].head()
check['changed'] = check['SMILES'] != check['canonical_smiles']
display(check)

## 1.4 — Missing Value Report

In [ ]:
import os
os.makedirs('../figures', exist_ok=True)

miss_report = missing_value_report(df, ENDPOINT_COLS)
print('Missing values per endpoint:')
display(miss_report)

# Bar chart
fig, ax = plt.subplots(figsize=(10, 4))
miss_report['pct_missing'].plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('% Missing per Endpoint')
ax.set_ylabel('% Missing')
ax.set_xlabel('')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../figures/1.2_missing_bar.png', dpi=150, bbox_inches='tight')
plt.show()

# Heatmap of missingness pattern
miss_matrix = df[ENDPOINT_COLS].isna().astype(int)
miss_matrix.columns = EP_SHORT

fig, ax = plt.subplots(figsize=(8, 3))
miss_corr = miss_matrix.corr()
sns.heatmap(miss_corr, annot=True, fmt='.2f', cmap='Oranges', ax=ax, vmin=0, vmax=1)
ax.set_title('Missingness Co-occurrence (correlation of NaN indicators)')
plt.tight_layout()
plt.savefig('../figures/1.2_missingness_cooccurrence.png', dpi=150, bbox_inches='tight')
plt.show()

HLM/RLM correlated in missingness -> Not all compounds tested so if a compound wasnt sent to the lab, its likely to be missing both not one or the other?

PPB_ H/R Same story, correlated as generally tested together?

#### 1.4b — Maximum Possible Correlation (WIP)

Upper bound on the Pearson r any model could achieve for each endpoint, given the assay's measurement noise.  
Uses the Brown, Muchmore & Hajduk simulation: adds Gaussian noise of magnitude `error` to the observed values and correlates against the originals over 1000 iterations.  
A model's r can be interpreted relative to this ceiling.

In [ ]:
mc_report = max_corr_report(df, ENDPOINT_COLS)
mc_report.index = EP_SHORT
print('Maximum possible R² per endpoint at 2x, 3x, 5x, 10x assay noise (Brown, Muchmore & Hajduk):')
display(mc_report.round(3))

## 1.5 — Summary Statistics

In [ ]:
desc = df[ENDPOINT_COLS].describe().T

skew = df[ENDPOINT_COLS].apply(lambda x: stats.skew(x.dropna()))
kurt = df[ENDPOINT_COLS].apply(lambda x: stats.kurtosis(x.dropna()))

desc['skewness'] = skew
desc['kurtosis'] = kurt
desc.index = EP_SHORT

print('Summary statistics (all 6 endpoints):')
display(desc[['count', 'mean', 'std', 'min', '25%', '50%', '75%', 'max']])

display(desc[['skewness', 'kurtosis']])

SOL only column with | skew | > 1

## 1.6 — Outlier Detection

Flag compounds >3σ from the mean per endpoint. Report only — no rows removed.

In [ ]:
outlier_summary = {}
outlier_indices = set()

for col in ENDPOINT_COLS:
    col_data = df[col].dropna()
    mu, sigma = col_data.mean(), col_data.std()
    mask = (df[col] - mu).abs() > 3 * sigma
    flagged = df.index[mask & df[col].notna()].tolist()
    outlier_summary[col] = len(flagged)
    outlier_indices.update(flagged)

print('Outlier counts per endpoint (>3σ):')
for k, v in outlier_summary.items():
    print(f"  {k}: {v}")
print(f"\nUnique compounds flagged in any endpoint: {len(outlier_indices)}")

if outlier_indices:
    print('\nFlagged rows (first 20):')
    display(df.loc[sorted(outlier_indices)[:20], ['Internal ID', 'SMILES'] + ENDPOINT_COLS])

Consider IQR outlier detection, better for skewed data

## 1.7 — Endpoint Distributions

In [ ]:
fig = endpoint_distributions(df, ENDPOINT_COLS)
plt.savefig('../figures/1.3_endpoint_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

**Distribution notes**:
- HLM_CLint: +ve skew, tail points right
- MDR1-MDCK ER: +ve skew
- SOLUBILITY: -ve skew
- PPB HUMAN: mild -ve skew
- PPB RAT: mild -ve skew
- RLM_CLint: No strong peak

**Floor-value spike in HLM_CLint and RLM_CLint**

The first bin spike visible in HLM_CLint and RLM_CLint is not a single outlier — it is a large cluster of compounds sharing the exact same minimum value:

| Endpoint | Floor value (log) | Compounds at floor | % of endpoint data |
|---|---|---|---|
| HLM_CLint | 0.676 | 958 | 31.0% |
| RLM_CLint | 1.028 | 346 | 11.3% |

The cause is currently unknown — possibilities include a lower limit of quantification when the measurements were taken in the lab.

**Why IQR does not flag it**: for HLM, Q1 equals the floor value (0.676) because >25% of the data sits there. The IQR lower bound is therefore −1.015 — well below any observed value — so no compound is flagged.

**Decision**: keep floor values as-is. All models see the same systematic pattern, so relative comparisons remain valid. The practical effect is that models partially learn to predict the floor for a large fraction of compounds, which may inflate apparent accuracy near that value.

## 1.8 — Endpoint Correlations

In [ ]:
corr = df[ENDPOINT_COLS].corr(method='spearman')
corr.index = corr.columns = EP_SHORT

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, vmin=-1, vmax=1, ax=ax, square=True
)
ax.set_title('Spearman Correlation — Endpoints')
plt.tight_layout()
plt.savefig('../figures/1.8_endpoint_correlation.png', dpi=150, bbox_inches='tight')
plt.show()

## 1.9 — Chemical Space Overview

In [ ]:
# Pre-validate SMILES before featurising — rdkit_descriptors raises on invalid input
assert validity["invalid_count"] == 0, f"Fix {validity['invalid_count']} invalid SMILES before featurising"

# Compute RDKit 2D descriptors (uses only valid SMILES)
desc_df = rdkit_descriptors(df['SMILES'].tolist())
desc_df.index = df.index
print('Descriptor shape:', desc_df.shape)
display(desc_df.describe())

MW = Molecular weight
LogP = Lipophilicity
TPSA = Topological polar surface area - absorption/permeability
HBD = H-bond donors
HBA = H-bond acceptors
RotBonds = Rotatable single bonds - flexibility

lipophiliicty - how fat loving molec is
tpa - tracks with molec weight§§

In [ ]:
# Histograms of physicochemical properties
prop_labels = {
    'MW': 'Molecular Weight (Da)',
    'LogP': 'LogP',
    'TPSA': 'TPSA (Å²)',
    'HBD': 'H-Bond Donors',
    'HBA': 'H-Bond Acceptors',
    'RotBonds': 'Rotatable Bonds',
}

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, (col, label) in zip(axes.flatten(), prop_labels.items()):
    data = desc_df[col].dropna()
    ax.hist(data, bins=40, color='teal', edgecolor='white', alpha=0.8)
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('Count')
plt.tight_layout()
plt.suptitle('Physicochemical Property Distributions', y=1.02)
plt.savefig('../figures/1.5_physicochemical_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Lipinski Ro5 violations (informational only)
lip_violations = (
    (desc_df['MW'] > 500).astype(int) +
    (desc_df['LogP'] > 5).astype(int) +
    (desc_df['HBD'] > 5).astype(int) +
    (desc_df['HBA'] > 10).astype(int)
)

print('Lipinski Ro5 violations per compound:')
print(lip_violations.value_counts().sort_index())
print(f"\nCompounds with >=2 violations (Ro5 fails): {(lip_violations >= 2).sum()} "
      f"({(lip_violations >= 2).mean()*100:.1f}%)")

heuristic, analysis of all drugs FDA approved, found trend that 5 rules that generally observable in drugs approved

3332 compounds break 0 rules, 148 break 1 rule etc

## 1.9b — Pairwise Chemical Similarity Distribution

How diverse is the compound library? If most compounds are very similar (high Tanimoto), the dataset is clustered around a few scaffolds; if similarity is low, the library covers a broad chemical space.

**Method**: Tanimoto similarity on ECFP4 Morgan fingerprints (radius=2, 2048 bits). For N compounds, there are N×(N−1)/2 unique pairs. RDKit's `BulkTanimotoSimilarity` computes all similarities of one fingerprint against a list in a single C++ call — much faster than a Python double loop.

**How Tanimoto works**: Each molecule is represented as a binary fingerprint (bit vector). The Tanimoto coefficient between two fingerprints A and B is:

$$T(A,B) = \frac{|A \cap B|}{|A \cup B|} = \frac{\text{bits on in both}}{\text{bits on in either}}$$

- T = 1.0 → identical fingerprints (same substructure features)
- T = 0.0 → no shared bits (completely different features)
- T ≈ 0.3–0.5 → typical for unrelated drug-like molecules

In [ ]:
# ── Small subsample demo: how BulkTanimotoSimilarity works ───────────────────
from rdkit.Chem import AllChem
from rdkit import Chem, DataStructs

# Pick 5 molecules to illustrate
demo_smiles = df['SMILES'].iloc[:5].tolist()
demo_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius=2, nBits=2048)
            for s in demo_smiles]

# BulkTanimotoSimilarity: compute similarity of fp[0] against ALL others in one C++ call
sims_to_first = DataStructs.BulkTanimotoSimilarity(demo_fps[0], demo_fps[1:])
print('Tanimoto similarities of compound 0 vs compounds 1–4:')
for i, sim in enumerate(sims_to_first, start=1):
    print(f'  0 vs {i}: {sim:.3f}')

# For all unique pairs (upper triangle), we loop i and use Bulk for each row
print(f'\nAll pairwise similarities (5 compounds, {5*4//2} unique pairs):')
for i in range(len(demo_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(demo_fps[i], demo_fps[i+1:])
    for j, sim in enumerate(sims, start=i+1):
        print(f'  {i} vs {j}: {sim:.3f}')

In [ ]:
# ── Full dataset pairwise similarity distribution ────────────────────────────
# 3521 compounds → 3521×3520/2 = 6,196,960 unique pairs

all_smiles = df['SMILES'].tolist()
all_fps = [AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(s), radius=2, nBits=2048)
           for s in all_smiles]
print(f'Computed {len(all_fps)} fingerprints')

# Collect all pairwise Tanimoto similarities using BulkTanimotoSimilarity
all_sims = []
for i in range(len(all_fps)):
    sims = DataStructs.BulkTanimotoSimilarity(all_fps[i], all_fps[i+1:])
    all_sims.extend(sims)

all_sims = np.array(all_sims)
print(f'Pairwise similarities: {len(all_sims):,} pairs')
print(f'Mean: {all_sims.mean():.3f}, Median: {np.median(all_sims):.3f}, '
      f'Std: {all_sims.std():.3f}')
print(f'Min: {all_sims.min():.3f}, Max: {all_sims.max():.3f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(all_sims, bins=100, color='steelblue', alpha=0.8, edgecolor='none')
axes[0].axvline(all_sims.mean(), color='red', linestyle='--', linewidth=1.5,
                label=f'Mean = {all_sims.mean():.3f}')
axes[0].axvline(np.median(all_sims), color='orange', linestyle='--', linewidth=1.5,
                label=f'Median = {np.median(all_sims):.3f}')
axes[0].set_xlabel('Tanimoto Similarity (ECFP4)')
axes[0].set_ylabel('Number of Pairs')
axes[0].set_title('Pairwise Tanimoto Similarity Distribution')
axes[0].legend()
axes[0].set_xlim(0, 1)

# Cumulative distribution
sorted_sims = np.sort(all_sims)
cdf = np.arange(1, len(sorted_sims) + 1) / len(sorted_sims)
axes[1].plot(sorted_sims, cdf, color='steelblue', linewidth=1.5)
axes[1].axhline(0.5, color='grey', linestyle=':', alpha=0.5)
axes[1].axvline(np.median(all_sims), color='orange', linestyle='--', linewidth=1,
                label=f'Median = {np.median(all_sims):.3f}')
axes[1].set_xlabel('Tanimoto Similarity (ECFP4)')
axes[1].set_ylabel('Cumulative Fraction of Pairs')
axes[1].set_title('CDF of Pairwise Similarity')
axes[1].legend()
axes[1].set_xlim(0, 1)

plt.tight_layout()
plt.savefig('../figures/1.6_pairwise_similarity.png', dpi=150, bbox_inches='tight')
plt.show()

# Summary statistics by threshold
for thresh in [0.3, 0.5, 0.7, 0.85]:
    frac = (all_sims >= thresh).mean() * 100
    print(f'Pairs with Tanimoto >= {thresh}: {frac:.1f}%')

## 1.10 — EDA Conclusions

> Findings are documented; cleaning strategy (SYNC-002) decided post-EDA review.

### Invalid SMILES
- **Finding**: 0 invalid SMILES out of 3521 (100% valid). No action required.

### Duplicates
- **Finding**: 0 duplicate raw SMILES; 0 duplicate canonical SMILES. Dataset is clean on this front.

### Missingness
- **Finding**: Missingness varies dramatically by endpoint:
  - HLM: 434 missing (12.3%), RLM: 467 missing (13.3%) — manageable
  - MDR1: 879 missing (25.0%), SOL: 1348 missing (38.3%) — substantial
  - PPB_HUMAN: 3327 missing (94.5%), PPB_RAT: 3353 missing (95.2%) — near-complete missingness
- **Cleaning strategy to decide (SYNC-002)**: PPB endpoints are nearly empty. Per-endpoint filtering (train each model on rows with non-missing values for that endpoint) is strongly preferred over complete-cases (which would leave only ~5% of data).

### Endpoint Distributions & Outliers
- **Skewness**: SOLUBILITY is left-skewed (skew=-1.64); HLM and MDR1 are mildly right-skewed (~0.6–0.8); RLM, PPB endpoints near-symmetric.
- **Outliers (>3σ)**: SOLUBILITY has 20 flagged outliers — largest concern. HLM: 3, MDR1: 1, PPB_RAT: 1. ~25 unique compounds flagged across all endpoints.
- **Endpoints to watch**: SOLUBILITY (high skew + most outliers + high missingness).

### Chemical Space
- **Drug-likeness**: 94.7% of compounds have 0 Lipinski Ro5 violations.
- **Violations**: 148 compounds have 1 violation, 41 have ≥2 (Ro5 fails).

## 1.10a — Stereoisomer Exclusion Check

Check for stereoisomer pairs with >3-fold difference in any log-scale endpoint. Following the filtering criterion in Fang et al. (ADME prospective validation paper), we flag and exclude any such pairs found.

**Why this matters**: Our ECFP4 fingerprints are computed with `useChirality=False` (the standard default), so the Morgan algorithm ignores stereochemistry annotations (R/S, E/Z). Two stereoisomers share the same atom connectivity and differ only in 3D spatial arrangement, producing **identical fingerprint bit vectors**. If they also differ substantially in activity, the model sees the same input for different target values — irresolvable label noise.

> **Why not set `useChirality=True`?** Stereochemistry annotations in SMILES are often incomplete or inconsistent across datasets; encoding unreliable chirality can introduce more noise than it resolves. The chirality-off default is standard practice in QSAR fingerprinting for this reason.

In [ ]:
from src.cleaning import exclude_stereoisomer_pairs

n_before = len(df)
df, excluded_stereo = exclude_stereoisomer_pairs(df, 'SMILES', ENDPOINT_COLS, fold_threshold=3)
n_after = len(df)

print(f'Stereoisomer exclusion (>3-fold difference in any log-scale endpoint):')
print(f'  Compounds before: {n_before}')
print(f'  Excluded:         {n_before - n_after}  (indices: {excluded_stereo})')
print(f'  Compounds after:  {n_after}')

## 1.11 — IQR Outlier Detection

Flag compounds outside [Q1 − 1.5·IQR, Q3 + 1.5·IQR] per endpoint.  
**Policy**: flag only — no rows removed. Outliers are kept for now, can look into how their exclusion later affects performance.  
Note: 3σ detection was used in Section 1.6 for reference; IQR is preferred here as it is more robust for skewed distributions (SOLUBILITY skewness = −1.64).

In [ ]:
from src.cleaning import flag_iqr_outliers, filter_endpoint

iqr_outlier_summary = {}

for col, short in zip(ENDPOINT_COLS, EP_SHORT):
    tag_col = f'iqr_outlier_{short}'
    df[tag_col] = False
    df_ep = filter_endpoint(df, col)
    mask = flag_iqr_outliers(df_ep, col)
    df.loc[mask[mask].index, tag_col] = True
    iqr_outlier_summary[col] = int(mask.sum())

print('IQR outlier counts per endpoint (1.5×IQR rule, flag only):')
for col, n in iqr_outlier_summary.items():
    print(f'  {col}: {n}')

print('\nOutlier tag columns added to df:', [f'iqr_outlier_{s}' for s in EP_SHORT])
print('\nExample — SOL outliers:')
display(df[df['iqr_outlier_SOL']][['Internal ID', 'SMILES', 'LOG SOLUBILITY PH 6.8 (ug/mL)']].head())

## 1.12 — Per-Endpoint Row Counts

Effective N for each model under per-endpoint filtering.  
PPB models (~170–190 rows) will have wider confidence intervals than HLM/RLM models (~3000 rows).

In [ ]:
from src.cleaning import filter_endpoint

header = f"{'Endpoint':<52} {'N_total':>7} {'N_missing':>10} {'N_for_model':>13} {'IQR_outliers':>13}"
print(header)
print('-' * len(header))

for col, short in zip(ENDPOINT_COLS, EP_SHORT):
    n_missing = int(df[col].isna().sum())
    n_model = len(filter_endpoint(df, col))
    n_iqr = iqr_outlier_summary[col]
    print(f'  {short:<50} {len(df):>7} {n_missing:>10} {n_model:>13} {n_iqr:>13}')

---

# Part 2 — Featurization & Baseline Models

**Goal**: Train 6 baseline regressors (MeanPredictor dummy + 5 real models) on all 6 ADME endpoints using ECFP4 Morgan fingerprints; evaluate with R², RMSE, MSE.  
**Models**: MeanPredictor (dummy baseline), Ridge, BayesianRidge, RandomForest, XGBoost, LightGBM  
**Featurization**: ECFP4 Morgan fingerprints (radius=2, 2048 bits) — see `src/features.morgan_fingerprints`

## 2.1 — Featurization

Generate ECFP4 Morgan fingerprints (radius=2, 2048 bits) for each per-endpoint filtered subset.

In [ ]:
from src.features import morgan_fingerprints
from src.cleaning import filter_endpoint

X_dict = {}  # {endpoint_col: (N, 2048) ndarray}
y_dict = {}  # {endpoint_col: (N,) ndarray}

for col in ENDPOINT_COLS:
    df_ep = filter_endpoint(df, col)
    X_dict[col] = morgan_fingerprints(df_ep['SMILES'].tolist())
    y_dict[col] = df_ep[col].values

for col in ENDPOINT_COLS:
    print(f"  {col}: X={X_dict[col].shape}, y={y_dict[col].shape}")

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors

# ── Pick first compound from the HLM subset ───────────────────────────────────
hlm_df = filter_endpoint(df, ENDPOINT_COLS[0])
smi = hlm_df.iloc[0]['SMILES']
mol = Chem.MolFromSmiles(smi)

fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
fp_arr = np.array(fp)
n_set = int(fp_arr.sum())

print(f"Compound : {hlm_df.iloc[0]['Internal ID']}")
print(f"SMILES   : {smi[:80]}{'…' if len(smi)>80 else ''}")
print(f"Bits ON  : {n_set} / 2048  ({n_set/2048*100:.1f}% density)")
print(f"First 10 ON-bit indices: {np.where(fp_arr)[0][:10].tolist()}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Left: molecule structure
img = Draw.MolToImage(mol, size=(420, 300))
axes[0].imshow(img)
axes[0].axis('off')
axes[0].set_title(f'{hlm_df.iloc[0]["Internal ID"]} — 2D structure', fontsize=10)

# Right: fingerprint as 32×64 grid (each cell = 1 bit)
fp_grid = fp_arr.reshape(32, 64)
im = axes[1].imshow(fp_grid, cmap='Blues', aspect='auto', interpolation='none', vmin=0, vmax=1)
axes[1].set_title(
    f'ECFP4 fingerprint  (2048 bits → 32 × 64 grid)\n'
    f'{n_set} bits ON (blue)  |  {2048-n_set} bits OFF (white)', fontsize=10)
axes[1].set_xlabel('Bit column (0–63)')
axes[1].set_ylabel('Bit row (× 64)')

plt.colorbar(im, ax=axes[1], shrink=0.7, label='bit value')
plt.suptitle('Morgan / ECFP4 fingerprint: molecule → 2048-bit binary vector', fontsize=11)
plt.tight_layout()
plt.show()

print("""
How it works (radius=2 → ECFP4):
  r=0  each atom is hashed individually (element, charge, …)
  r=1  each atom + its direct neighbours
  r=2  each atom + all atoms within 2 bonds     ← ECFP4 uses this
  Each circular environment is hashed to one of the 2048 bit positions.
  Two different environments that hash to the same bit → bit collision (rare at 2048 bits).
""")

In [ ]:
# ── Part A: how radius controls what each bit "sees" ─────────────────────────
# Use aspirin — small enough to reason about atom-by-atom
from rdkit import Chem
from rdkit.Chem import Draw, rdMolDescriptors

demo_smi = 'CC(=O)Oc1ccccc1C(=O)O'   # aspirin
demo_mol = Chem.MolFromSmiles(demo_smi)

print("Aspirin:", demo_smi)
print()
print("radius  ECFP name   bits ON   interpretation")
print("──────  ─────────   ───────   ──────────────")
for r in [0, 1, 2, 3]:
    fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(demo_mol, radius=r, nBits=2048)
    n = int(sum(fp))
    desc = {
        0: "each atom alone (element, charge, degree…)",
        1: "each atom + its direct bonds",
        2: "each atom + atoms within 2 bonds  ← ECFP4",
        3: "each atom + atoms within 3 bonds  ← ECFP6",
    }[r]
    print(f"  r={r}    ECFP{r*2}       {n:>4}     {desc}")

print()
print("Going deeper in radius → more context captured per bit → bits grow slowly")
print("(overlapping environments share atoms, so bits don't double each step)")

In [ ]:
# ── Part B: draw the actual circular environments for each ON bit ─────────────
# DrawMorganBit has a C++ API mismatch in this RDKit build, so we draw manually:
# FindAtomEnvironmentOfRadiusN gives us the bonds in the circular neighbourhood,
# then MolToImage highlights those atoms directly.

def draw_morgan_env(mol, centre_atom, radius, size=(200, 160)):
    """Highlight the Morgan environment of `centre_atom` at `radius`."""
    bond_ids = Chem.FindAtomEnvironmentOfRadiusN(mol, radius, centre_atom)
    env_atoms = {centre_atom}
    for bid in bond_ids:
        b = mol.GetBondWithIdx(bid)
        env_atoms.add(b.GetBeginAtomIdx())
        env_atoms.add(b.GetEndAtomIdx())

    colors = {a: (1.0, 0.85, 0.0) if a == centre_atom   # yellow  = centre
                 else (0.6, 0.8, 1.0)                    # blue    = neighbourhood
              for a in env_atoms}
    return Draw.MolToImage(mol, size=size,
                           highlightAtoms=list(env_atoms),
                           highlightAtomColors=colors)

bi = {}
rdMolDescriptors.GetMorganFingerprintAsBitVect(demo_mol, radius=2, nBits=2048, bitInfo=bi)
on_bits = sorted(bi.keys())
print(f"Aspirin has {len(on_bits)} ON bits at r=2.  Each row below is one bit:\n")

for bit in on_bits:
    atom_idx, radius = bi[bit][0]
    atom_sym = demo_mol.GetAtomWithIdx(atom_idx).GetSymbol()
    print(f"  bit {bit:4d}  centre=atom{atom_idx}({atom_sym})  radius={radius}")

n_cols = 5
n_rows = -(-len(on_bits) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2.5, n_rows * 2.2))
axes = axes.flatten()

for ax, bit in zip(axes, on_bits):
    atom_idx, radius = bi[bit][0]
    atom_sym = demo_mol.GetAtomWithIdx(atom_idx).GetSymbol()
    img = draw_morgan_env(demo_mol, atom_idx, radius, size=(200, 160))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f'bit {bit}\natom {atom_idx} ({atom_sym}), r={radius}', fontsize=7.5)

for ax in axes[len(on_bits):]:
    ax.axis('off')

plt.suptitle(
    "Every ON bit in aspirin's ECFP4 fingerprint\n"
    "Yellow = centre atom | blue = atoms within radius",
    fontsize=10)
plt.tight_layout()
plt.show()

print("Key insight: each bit = one unique chemical neighbourhood.")
print("The SAME bit fires for ANY molecule that contains that same neighbourhood.")

In [ ]:
# ── Part C: Tanimoto similarity — how fingerprints measure molecular similarity ─
from rdkit.DataStructs import TanimotoSimilarity

molecules = [
    ('Aspirin',              'CC(=O)Oc1ccccc1C(=O)O'),
    ('Salicylic acid\n(aspirin −acetyl)', 'OC(=O)c1ccccc1O'),
    ('Ibuprofen\n(different scaffold)',   'CC(C)Cc1ccc(cc1)C(C)C(=O)O'),
    ('Caffeine\n(very different)',        'Cn1cnc2c1c(=O)n(C)c(=O)n2C'),
]

ref_name, ref_smi = molecules[0]
ref_fp = rdMolDescriptors.GetMorganFingerprintAsBitVect(
    Chem.MolFromSmiles(ref_smi), radius=2, nBits=2048)
ref_on = int(sum(ref_fp))

print(f"Reference: {ref_name}  ({ref_on} bits ON)\n")
print(f"{'Molecule':<30} {'bits ON':>7}  {'shared |A∩B|':>13}  {'union |A∪B|':>12}  {'Tanimoto':>9}")
print("─" * 78)

fps = []
for name, smi in molecules:
    mol_ = Chem.MolFromSmiles(smi)
    fp_ = rdMolDescriptors.GetMorganFingerprintAsBitVect(mol_, radius=2, nBits=2048)
    fps.append(fp_)
    n_on   = int(sum(fp_))
    shared = int(sum(a and b for a, b in zip(ref_fp, fp_)))
    union  = ref_on + n_on - shared
    sim    = TanimotoSimilarity(ref_fp, fp_)
    label  = name.replace('\n', ' ')
    print(f"  {label:<28} {n_on:>7}  {shared:>13}  {union:>12}  {sim:>9.3f}")

print()
print("Tanimoto = |A ∩ B| / |A ∪ B|  =  shared / (bits_A + bits_B - shared)")
print("Range 0 (nothing in common) → 1 (identical fingerprint)")

# Draw all four molecules with their similarity scores
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, (name, smi), fp_ in zip(axes, molecules, fps):
    mol_ = Chem.MolFromSmiles(smi)
    img = Draw.MolToImage(mol_, size=(300, 210))
    ax.imshow(img)
    ax.axis('off')
    sim    = TanimotoSimilarity(ref_fp, fp_)
    shared = int(sum(a and b for a, b in zip(ref_fp, fp_)))
    n_on   = int(sum(fp_))
    ax.set_title(f'{name}\nTanimoto = {sim:.3f}\n{shared} shared / {ref_on+n_on-shared} union bits',
                 fontsize=8.5)

plt.suptitle('Tanimoto similarity via ECFP4: shared circular environments → higher score', fontsize=10)
plt.tight_layout()
plt.show()

## 2.2 — Train/Test Split

80/20 random split per endpoint (seed=42 for reproducibility).

In [ ]:
from sklearn.model_selection import train_test_split
from src.cleaning import filter_endpoint

splits = {}  # {col: (X_train, X_test, y_train, y_test, smiles_train, smiles_test)}

for col in ENDPOINT_COLS:
    df_ep = filter_endpoint(df, col)
    X, y = X_dict[col], y_dict[col]
    smiles_all = df_ep['SMILES'].tolist()
    X_train, X_test, y_train, y_test, smiles_train, smiles_test = train_test_split(
        X, y, smiles_all, test_size=0.2, random_state=SEED
    )
    splits[col] = (X_train, X_test, y_train, y_test, smiles_train, smiles_test)
    print(f"  {col}: train={len(y_train)}, test={len(y_test)}")

## 2.3 — Model Training & Evaluation

In [ ]:
from src.models import get_baseline_models, evaluate_model

results = []
preds_store = {}  # {col: {model_name: (y_test, y_pred)}}

for col in ENDPOINT_COLS:
    X_train, X_test, y_train, y_test, _, _ = splits[col]
    preds_store[col] = {}
    for name, model in get_baseline_models().items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        preds_store[col][name] = (y_test, y_pred)
        metrics = evaluate_model(model, X_test, y_test, y_pred=y_pred)
        results.append({"endpoint": col, "model": name, "features": "ECFP4", **metrics})

results_df = pd.DataFrame(results)
print("Training complete.")
print(f"\nMean R² across endpoints per model:")
print(results_df.groupby('model')['R2'].mean().round(3).sort_values(ascending=False).to_string())

## 2.3b — Baselines on RDKit 2D Descriptors

Same 6 baseline models, same train/test splits, but featurized with the full 200-descriptor RDKit 2D normalized set (via `descriptastorus`) instead of Morgan ECFP4. This is the same descriptor set that MPNN2 uses as auxiliary features.

In [ ]:
from src.features import rdkit_2d_features
from src.models import get_baseline_models, evaluate_model

# ── Compute RDKit 2D features for each endpoint ───────────────────────────────
X_rdkit_dict = {}
for col in ENDPOINT_COLS:
    df_ep = filter_endpoint(df, col)
    X_rdkit_dict[col] = rdkit_2d_features(df_ep['SMILES'].tolist())
    print(f"  {col}: X_rdkit={X_rdkit_dict[col].shape}")

# ── Split using same random_state → identical molecule partitions ──────────────
splits_rdkit = {}
for col in ENDPOINT_COLS:
    df_ep = filter_endpoint(df, col)
    X_rdkit = X_rdkit_dict[col]
    y = y_dict[col]
    smiles_all = df_ep['SMILES'].tolist()
    X_train, X_test, y_train, y_test, smi_tr, smi_te = train_test_split(
        X_rdkit, y, smiles_all, test_size=0.2, random_state=SEED
    )
    splits_rdkit[col] = (X_train, X_test, y_train, y_test, smi_tr, smi_te)
    # Sanity check: same molecules in train/test as Morgan split
    assert np.array_equal(y_test, splits[col][3]), f"Split mismatch for {col}!"

# ── Train all 6 baselines on RDKit 2D features ────────────────────────────────
results_rdkit = []
preds_store_rdkit = {}

for col in ENDPOINT_COLS:
    X_train, X_test, y_train, y_test, _, _ = splits_rdkit[col]
    preds_store_rdkit[col] = {}
    for name, model in get_baseline_models().items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        preds_store_rdkit[col][name] = (y_test, y_pred)
        metrics = evaluate_model(model, X_test, y_test, y_pred=y_pred)
        results_rdkit.append({"endpoint": col, "model": name, "features": "RDKit2D", **metrics})

# ── Merge into results_df ─────────────────────────────────────────────────────
results_df = pd.DataFrame(results + results_rdkit)
results_df['ep_short'] = results_df['endpoint'].map(EP_SHORT_MAP)

print("RDKit 2D training complete.")
print(f"\nMean R² across endpoints — ECFP4 vs RDKit2D:")
print(results_df.groupby(['features', 'model'])['R2'].mean().round(3).unstack('features').to_string())

## 2.4 — Results Summary

In [ ]:
results_df['ep_short'] = results_df['endpoint'].map(EP_SHORT_MAP)

for feat in results_df['features'].unique():
    subset = results_df[results_df['features'] == feat]
    print(f'R² (test set) — {feat}:')
    r2_table = subset.pivot(index='ep_short', columns='model', values='R2').round(3)
    display(r2_table)
    print(f'\nRMSE (test set) — {feat}:')
    rmse_table = subset.pivot(index='ep_short', columns='model', values='RMSE').round(3)
    display(rmse_table)
    print()

Notes:

1. Ridge performs poorly on ECFP4, better on RDKit2D, bayesian ridge has regularization that results in sparse binary inputs not being so much of an issue
2. RDKit2D > ECFP4 Wonder why? Whats the main diff between the 2 featurization approaches?
3. Tree based methods surpass BayesianRidge

Note 2:
- ECFP4: binary circular fingerprint — encodes which structural fragments are present (topology). 2048 sparse binary bits. Captures "what the molecule looks like structurally."
- RDKit2D: ~200 continuous physicochemical descriptors — MW, LogP, TPSA, H-bond donors/acceptors, rotatable bonds, ring counts, etc.


## 2.4b — Fingerprint Radius Sensitivity (r=1, 2, 4)

Same 5 baseline models, same 80/20 split, same endpoints — only the Morgan fingerprint radius varies. Radius controls how large a circular neighbourhood each bit encodes: r=1 captures immediate bonds, r=2 (ECFP4, our default) extends two bonds out, r=4 (ECFP8) captures larger structural contexts. MeanPredictor excluded as it ignores fingerprints entirely.

In [ ]:
if not RUN_RADIUS_SENSITIVITY:
    print('Radius sensitivity skipped (RUN_RADIUS_SENSITIVITY=False).')
else:
    from src.features import morgan_fingerprints
    from src.cleaning import filter_endpoint
    from sklearn.model_selection import train_test_split

    RADII = [1, 2, 4]
    radius_rows = []

    for radius in RADII:
        for col in ENDPOINT_COLS:
            df_ep = filter_endpoint(df, col)
            X_r = morgan_fingerprints(df_ep['SMILES'].tolist(), radius=radius)
            y_r = df_ep[col].values
            X_train, X_test, y_train, y_test = train_test_split(X_r, y_r, test_size=0.2, random_state=SEED)
            for name, model in get_baseline_models().items():
                model.fit(X_train, y_train)
                metrics = evaluate_model(model, X_test, y_test)
                radius_rows.append({'radius': radius, 'endpoint': col,
                                     'ep_short': EP_SHORT_MAP[col], 'model': name, **metrics})

    radius_df = pd.DataFrame(radius_rows)
    print(f'Radius sensitivity: {len(radius_df)} rows ({len(RADII)} radii × {len(ENDPOINT_COLS)} endpoints × 6 models)')

In [ ]:
if not RUN_RADIUS_SENSITIVITY:
    print('Radius sensitivity plot skipped.')
else:
    real_models = ['Ridge', 'BayesianRidge', 'RandomForest', 'XGBoost', 'LightGBM']
    radius_colors = {1: '#e05c5c', 2: '#4878cf', 4: '#4caf6e'}
    x = np.arange(len(real_models))
    bar_width = 0.25

    fig, axes = plt.subplots(2, 3, figsize=(16, 9), sharey=False)
    axes = axes.flatten()

    for ax, ep in zip(axes, EP_SHORT):
        ep_df = radius_df[(radius_df['ep_short'] == ep) & (radius_df['model'].isin(real_models))]
        for i, radius in enumerate(RADII):
            r_df = ep_df[ep_df['radius'] == radius]
            r2_vals = [r_df[r_df['model'] == m]['R2'].mean() for m in real_models]
            ax.bar(x + i * bar_width, r2_vals, bar_width,
                   label=f'radius={radius}', color=radius_colors[radius], alpha=0.85)
        ax.set_title(ep)
        ax.set_xticks(x + bar_width)
        ax.set_xticklabels(real_models, rotation=25, ha='right', fontsize=8)
        ax.set_ylabel('R²')
        ax.axhline(0, color='black', linewidth=0.7, linestyle='--')
        ax.grid(True, alpha=0.3, axis='y')

    axes[0].legend(fontsize=8)
    fig.suptitle('R² by Morgan Fingerprint Radius (1, 2, 4) — ADME Endpoints', fontsize=12)
    plt.tight_layout()
    os.makedirs('../figures', exist_ok=True)
    fig.savefig('../figures/2.4b_radius_sensitivity.png', dpi=150, bbox_inches='tight')
    plt.show()

- BayesianRidge marginally better than LightGBM, much better than other models

## 2.5 — Predicted vs Actual Plots

In [ ]:
from src.plotting import pred_vs_actual_grid

# Note if using preds_store hten using morgan fingerprints, need to swap to ECFP4 Morgan fingerprints

for col in ENDPOINT_COLS:
    ep = EP_SHORT_MAP[col]
    fig = pred_vs_actual_grid(preds_store_rdkit[col], title=ep)
    plt.savefig(f'../figures/2.5_pred_vs_actual_{ep}.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
from src.plotting import pred_vs_actual_grid

# Note if using preds_store hten using morgan fingerprints, need to swap to ECFP4 Morgan fingerprints

for col in ENDPOINT_COLS:
    ep = EP_SHORT_MAP[col]
    fig = pred_vs_actual_grid(preds_store[col], title=ep)
    plt.savefig(f'../figures/2.5_pred_vs_actual_{ep}.png', dpi=150, bbox_inches='tight')
    plt.show()

## 2.6 — MPNN2 (ChemProp: molecular graph + RDKit 2D descriptors)

MPNN2 - a message-passing neural network trained on the molecular graph with RDKit 2D normalised descriptors as additional features (ChemProp 1.6.1). This provides a deep learning comparison point against the sklearn baselines above.

In [ ]:
if not TRAIN_MPNN2:
    print('MPNN2 training skipped (TRAIN_MPNN2=False). Set to True in the cell above to run.')
else:
    import os
    import csv
    from sklearn.metrics import r2_score, mean_squared_error as mse_fn
    from chemprop.args import TrainArgs, PredictArgs
    from chemprop.train import cross_validate, run_training, make_predictions

    MPNN2_DIR = os.path.join('..', 'models', 'mpnn2')

    def _write_csv(path, smiles_list, targets=None):
        """Write ChemProp-format CSV: smiles column + optional target column."""
        with open(path, 'w', newline='') as f:
            writer = csv.writer(f)
            if targets is not None:
                writer.writerow(['smiles', 'target'])
                for smi, y in zip(smiles_list, targets):
                    writer.writerow([smi, float(y)])
            else:
                writer.writerow(['smiles'])
                for smi in smiles_list:
                    writer.writerow([smi])

    mpnn2_results = []
    mpnn2_preds_store = {}  # {col: (y_test, y_pred)}

    for col in ENDPOINT_COLS:
        ep = EP_SHORT_MAP[col]
        _, _, y_train, y_test, smiles_train, smiles_test = splits[col]

        ep_dir    = os.path.join(MPNN2_DIR, ep)
        train_csv = os.path.join(ep_dir, 'train.csv')
        test_csv  = os.path.join(ep_dir, 'test.csv')
        preds_csv = os.path.join(ep_dir, 'preds.csv')
        os.makedirs(ep_dir, exist_ok=True)

        _write_csv(train_csv, smiles_train, y_train)
        _write_csv(test_csv,  smiles_test)

        train_args = TrainArgs().parse_args([
            '--data_path',          train_csv,
            '--dataset_type',       'regression',
            '--save_dir',           ep_dir,
            '--features_generator', 'rdkit_2d_normalized',
            '--no_features_scaling',
            '--metric',             'mae',
            '--epochs',             '30',
            '--num_folds',          '1',
            '--quiet',
        ])
        cross_validate(args=train_args, train_func=run_training)

        pred_args = PredictArgs().parse_args([
            '--test_path',          test_csv,
            '--preds_path',         preds_csv,
            '--checkpoint_dir',     ep_dir,
            '--features_generator', 'rdkit_2d_normalized',
            '--no_features_scaling',
        ])
        raw_preds = make_predictions(args=pred_args)
        y_pred = np.array([p[0] for p in raw_preds])

        mpnn2_preds_store[col] = (y_test, y_pred)

        r2   = r2_score(y_test, y_pred)
        rmse = float(np.sqrt(mse_fn(y_test, y_pred)))
        mpnn2_results.append({'endpoint': col, 'ep_short': ep, 'model': 'MPNN2', 'R2': r2, 'RMSE': rmse, 'MSE': rmse**2})
        print(f'{ep:6s}  R²={r2:+.3f}  RMSE={rmse:.3f}')

## 2.6b — MPNN (Graph-Only, no RDKit descriptors)

Same ChemProp architecture as MPNN2 above but **without** the `rdkit_2d_normalized` auxiliary features. This isolates the learned graph representation to measure whether adding handcrafted descriptors helps the neural network.

In [ ]:
if not TRAIN_MPNN_GRAPH:
    print('MPNN (graph-only) training skipped (TRAIN_MPNN_GRAPH=False). Set to True above to run.')
else:
    import os
    import csv
    from sklearn.metrics import r2_score, mean_squared_error as mse_fn
    from chemprop.args import TrainArgs, PredictArgs
    from chemprop.train import cross_validate, run_training, make_predictions

    MPNN_GRAPH_DIR = os.path.join('..', 'models', 'mpnn_graph')

    def _write_csv(path, smiles_list, targets=None):
        """Write ChemProp-format CSV: smiles column + optional target column."""
        with open(path, 'w', newline='') as f:
            writer = csv.writer(f)
            if targets is not None:
                writer.writerow(['smiles', 'target'])
                for smi, y in zip(smiles_list, targets):
                    writer.writerow([smi, float(y)])
            else:
                writer.writerow(['smiles'])
                for smi in smiles_list:
                    writer.writerow([smi])

    mpnn_graph_results = []

    for col in ENDPOINT_COLS:
        ep = EP_SHORT_MAP[col]
        _, _, y_train, y_test, smiles_train, smiles_test = splits[col]

        ep_dir    = os.path.join(MPNN_GRAPH_DIR, ep)
        train_csv = os.path.join(ep_dir, 'train.csv')
        test_csv  = os.path.join(ep_dir, 'test.csv')
        preds_csv = os.path.join(ep_dir, 'preds.csv')
        os.makedirs(ep_dir, exist_ok=True)

        _write_csv(train_csv, smiles_train, y_train)
        _write_csv(test_csv, smiles_test)

        # Graph-only: NO --features_generator, NO --no_features_scaling
        train_args = TrainArgs().parse_args([
            '--data_path',     train_csv,
            '--dataset_type',  'regression',
            '--save_dir',      ep_dir,
            '--metric',        'mae',
            '--epochs',        '30',
            '--num_folds',     '1',
            '--quiet',
        ])
        cross_validate(args=train_args, train_func=run_training)

        pred_args = PredictArgs().parse_args([
            '--test_path',      test_csv,
            '--preds_path',     preds_csv,
            '--checkpoint_dir', ep_dir,
        ])
        raw_preds = make_predictions(args=pred_args)
        y_pred = np.array([p[0] for p in raw_preds])

        r2   = r2_score(y_test, y_pred)
        rmse = np.sqrt(mse_fn(y_test, y_pred))
        print(f"  {ep:6s}  R²={r2:.3f}  RMSE={rmse:.3f}")

        mpnn_graph_results.append({
            'endpoint': col, 'ep_short': ep, 'model': 'MPNN',
            'features': 'graph', 'R2': r2, 'RMSE': rmse, 'MSE': rmse**2,
        })

    mpnn_graph_df = pd.DataFrame(mpnn_graph_results)
    print('\nMPNN (graph-only) results:')
    display(mpnn_graph_df[['ep_short', 'R2', 'RMSE']].round(3))

In [ ]:
col_order_base = ['MeanPredictor', 'Ridge', 'BayesianRidge', 'RandomForest', 'XGBoost', 'LightGBM']
mpnn_cols = (['MPNN2'] if TRAIN_MPNN2 else []) + (['MPNN'] if TRAIN_MPNN_GRAPH else [])

for feat in results_df['features'].unique():
    subset = results_df[results_df['features'] == feat].copy()

    if TRAIN_MPNN2:
        subset = pd.concat([subset, pd.DataFrame(mpnn2_results)], ignore_index=True)
    if TRAIN_MPNN_GRAPH:
        subset = pd.concat([subset, mpnn_graph_df], ignore_index=True)

    col_order = [c for c in col_order_base + mpnn_cols if c in subset['model'].values]

    print(f'R² (test set) — {feat} + deep learning if included as part of training:')
    display(subset.pivot(index='ep_short', columns='model', values='R2').round(3)[col_order])

    print(f'\nRMSE (test set) — {feat} + deep learning if included as part of training:')
    display(subset.pivot(index='ep_short', columns='model', values='RMSE').round(3)[col_order])
    print()

## 2.7 — Similarity-Binned Prediction Error (Sørensen–Dice / FCFP, radius=4)

For each test compound, compute the mean Sørensen–Dice similarity to the top-5 nearest training neighbours using FCFP4 fingerprints (radius=4, 1024 bits), then bin test compounds by similarity and plot MAE per bin. This follows the methodology of Fang et al. to assess whether models perform worse on compounds that are structurally distant from the training set.

In [ ]:
from rdkit.Chem import AllChem
from rdkit import DataStructs

def _fcfp4_fps(smiles_list, n_bits=1024):
    """Compute FCFP fingerprints (feature-class Morgan, radius=4, 1024-bit) as RDKit ExplicitBitVect objects."""
    fps = []
    for smi in smiles_list:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            raise ValueError(f"Invalid SMILES: {smi!r}")
        fps.append(AllChem.GetMorganFingerprintAsBitVect(mol, radius=4, nBits=n_bits, useFeatures=True))
    return fps

TOP_K = 5  # mean of top-5 similarity, as per Fang et al.

sim_records = []

for col in ENDPOINT_COLS:
    _, _, _, _, smiles_train, smiles_test = splits[col]

    train_fps = _fcfp4_fps(smiles_train)
    test_fps  = _fcfp4_fps(smiles_test)

    mean_top5_sims = np.array([
        np.mean(sorted(DataStructs.BulkDiceSimilarity(fp, train_fps), reverse=True)[:TOP_K])
        for fp in test_fps
    ])
    sim_bins = np.clip(np.floor(mean_top5_sims * 10) / 10, 0.0, 0.9)

    # Baseline models
    for model_name, (y_test, y_pred) in preds_store[col].items():
        abs_errors = np.abs(y_test - y_pred)
        for sim_bin, abs_err in zip(sim_bins, abs_errors):
            sim_records.append({'ep_short': EP_SHORT_MAP[col], 'model': model_name,
                                 'sim_bin': sim_bin, 'abs_error': abs_err})
 
    # MPNN2 — included only if training was run
    if TRAIN_MPNN2:
        y_test_m, y_pred_m = mpnn2_preds_store[col]
        abs_errors = np.abs(y_test_m - y_pred_m)
        for sim_bin, abs_err in zip(sim_bins, abs_errors):
            sim_records.append({'ep_short': EP_SHORT_MAP[col], 'model': 'MPNN2',
                                 'sim_bin': sim_bin, 'abs_error': abs_err})

sim_df = pd.DataFrame(sim_records)
print(f'Records: {len(sim_df)}')
print(f"Models in sim_df: {sorted(sim_df['model'].unique())}")
print(f"Similarity bin distribution:\n{sim_df['sim_bin'].value_counts().sort_index().to_string()}")

In [ ]:
model_names = ['MeanPredictor', 'Ridge', 'BayesianRidge', 'RandomForest', 'XGBoost', 'LightGBM', 'MPNN2']
present_models = [m for m in model_names if m in sim_df['model'].unique()]
focus_eps   = ['HLM', 'MDR1']

bins       = np.arange(0.0, 1.0, 0.1)
bar_width  = 0.04
bar_colors = plt.cm.Greys(np.linspace(0.8, 0.3, len(bins)))

for ep in focus_eps:
    ep_df = sim_df[sim_df['ep_short'] == ep]
    n_models = len(present_models)
    fig, axes = plt.subplots(2, 4 if n_models > 6 else 3, figsize=(18 if n_models > 6 else 16, 9), sharey=True)
    axes = axes.flatten()

    for ax, model in zip(axes, present_models):
        m_df = ep_df[ep_df['model'] == model]
        stats = m_df.groupby('sim_bin')['abs_error'].agg(['mean', 'std', 'count'])

        for j, (sim_bin, row) in enumerate(stats.iterrows()):
            ax.bar(j, row['mean'], color=bar_colors[int(round(sim_bin * 10))],
                   width=0.7, alpha=0.95)
            ax.errorbar(j, row['mean'], yerr=row['std'], fmt='none',
                        color='black', capsize=3, linewidth=1)

        ax.set_title(model, fontsize=10)
        ax.set_xlabel('Mean top-5 Dice similarity to training set', fontsize=8)
        ax.set_ylabel('Mean absolute error', fontsize=8)
        ax.set_xticks(range(len(stats)))
        ax.set_xticklabels([f'({b:.1f},{b+0.1:.1f}]' for b in stats.index],
                           rotation=45, ha='right', fontsize=7)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.3, axis='y')

    for ax in axes[n_models:]:
        ax.set_visible(False)

    fig.suptitle(f'Similarity-Binned MAE — {ep} (FCFP radius=4, Dice, mean top-5, 0.1 bins)', fontsize=13)
    plt.tight_layout()
    plt.savefig(f'../figures/2.7_sim_binned_mae_{ep}.png', dpi=150, bbox_inches='tight')
    plt.show()

---

# Part 3 — Hyperparameter Tuning (Clean-Data Reference)

Tune LightGBM, RandomForest, and MPNN2 on each endpoint's **full, clean** training set. This serves two purposes:

1. **Validate the tuning pipeline** — confirm that RandomizedSearchCV and early stopping work correctly before the experiment loop.
2. **Establish a clean-data ceiling** — the best R² achievable with tuned hyperparameters on clean, full data. Phases 4–6 will show how performance degrades from this ceiling.

In Phases 4–6, tuning is **re-run per condition** (the tuned arm), alongside a baseline arm using default hyperparameters. This shows whether tuning buys resilience under degraded data conditions.

**Endpoints**: HLM, MDR1, SOL, RLM (4 modelling endpoints — PPB excluded, too sparse for CV-based tuning).

**CV scoring**: MAE (`neg_mean_absolute_error`) — less sensitive to noisy labels than RMSE, important for the noise injection experiments in Phase 5.

**MPNN2**: Fixed architecture + early stopping with MAE metric (controlled by `TUNE_MPNN2` flag).

## 3.1 — LightGBM Hyperparameter Tuning

In [ ]:
import os
from sklearn.model_selection import train_test_split
from src.tuning import tune_lightgbm, tune_rf, save_params, load_params, make_model
from src.models import evaluate_model

TUNED_PARAMS_DIR = os.path.join('..', 'models', 'tuned_params')
os.makedirs(TUNED_PARAMS_DIR, exist_ok=True)

# Focus modelling on 4 endpoints with sufficient data
MODEL_ENDPOINTS = [
    'LOG HLM_CLint (mL/min/kg)',
    'LOG MDR1-MDCK ER (B-A/A-B)',
    'LOG SOLUBILITY PH 6.8 (ug/mL)',
    'LOG RLM_CLint (mL/min/kg)',
]

lgbm_tuned_results = []

for col in MODEL_ENDPOINTS:
    ep = EP_SHORT_MAP[col]
    X_train, X_test, y_train, y_test, _, _ = splits_rdkit[col]

    # Hold out 20% of train as a fixed validation set for hyperparameter selection
    X_train_tune, X_val, y_train_tune, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=SEED
    )

    model, params = tune_lightgbm(X_train_tune, y_train_tune, X_val, y_val, n_iter=50, random_state=SEED)
    save_params(params, os.path.join(TUNED_PARAMS_DIR, f'{ep}_lgbm.json'))

    metrics = evaluate_model(model, X_test, y_test)
    lgbm_tuned_results.append({'ep_short': ep, 'model': 'LightGBM (tuned)', **metrics})
    print(f'{ep:6s}  R²={metrics["R2"]:+.3f}  RMSE={metrics["RMSE"]:.3f}  params={params}')

lgbm_tuned_df = pd.DataFrame(lgbm_tuned_results)

## 3.1b — Random Forest Hyperparameter Tuning

In [ ]:
rf_tuned_results = []

for col in MODEL_ENDPOINTS:
    ep = EP_SHORT_MAP[col]
    X_train, X_test, y_train, y_test, _, _ = splits_rdkit[col]

    # Hold out 20% of train as a fixed validation set for hyperparameter selection
    X_train_tune, X_val, y_train_tune, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=SEED
    )

    model, params = tune_rf(X_train_tune, y_train_tune, X_val, y_val, n_iter=50, random_state=SEED)
    save_params(params, os.path.join(TUNED_PARAMS_DIR, f'{ep}_rf.json'))

    metrics = evaluate_model(model, X_test, y_test)
    rf_tuned_results.append({'ep_short': ep, 'model': 'RandomForest (tuned)', **metrics})
    print(f'{ep:6s}  R²={metrics["R2"]:+.3f}  RMSE={metrics["RMSE"]:.3f}  params={params}')

rf_tuned_df = pd.DataFrame(rf_tuned_results)

### 3.1c — Baseline vs Tuned                                                                                                              

In [ ]:
# Compare baseline vs tuned for LightGBM, RF; BayesianRidge included as non-tuned reference
# Filter to RDKit2D only — tuning was performed on RDKit2D features
baseline = results_df[
    (results_df['features'] == 'RDKit2D') &
    (results_df['model'].isin(['LightGBM', 'RandomForest', 'BayesianRidge'])) &
    (results_df['ep_short'].isin(['HLM', 'MDR1', 'SOL', 'RLM']))
][['ep_short', 'model', 'R2', 'RMSE']].copy()

tuned = pd.concat([lgbm_tuned_df, rf_tuned_df])[['ep_short', 'model', 'R2', 'RMSE']]
all_compare = pd.concat([baseline, tuned], ignore_index=True)

col_order = ['BayesianRidge', 'RandomForest', 'RandomForest (tuned)', 'LightGBM', 'LightGBM (tuned)']

print('R² — Baseline vs Tuned:')
r2_pivot = all_compare.pivot(index='ep_short', columns='model', values='R2')[col_order].round(4)
display(r2_pivot)

print('\nRMSE — Baseline vs Tuned:')
rmse_pivot = all_compare.pivot(index='ep_short', columns='model', values='RMSE')[col_order].round(4)
display(rmse_pivot)

## 3.2 — MPNN2 Tuning (Early Stopping)

ChemProp's built-in early stopping handles tuning: fixed architecture (`hidden_size=300, depth=3`, ChemProp defaults), generous `--epochs 50`. The internal train/val split stops training when validation RMSE plateaus. Large endpoints only (HLM, MDR1, RLM, SOL; ≥500 training rows).

Set `TUNE_MPNN2 = True` to run in the first cell of the notebook

In [ ]:
if not TUNE_MPNN2:
    print('MPNN2 tuning skipped (TUNE_MPNN2=False). Set to True to run.')
else:
    import csv
    from chemprop.args import TrainArgs, PredictArgs
    from chemprop.train import cross_validate, run_training, make_predictions
    from sklearn.metrics import r2_score, mean_squared_error as mse_fn

    LARGE_ENDPOINTS = [
        'LOG HLM_CLint (mL/min/kg)',
        'LOG MDR1-MDCK ER (B-A/A-B)',
        'LOG SOLUBILITY PH 6.8 (ug/mL)',
        'LOG RLM_CLint (mL/min/kg)',
    ]

    MPNN2_TUNE_DIR = os.path.join('..', 'models', 'mpnn2_tuned')
    MPNN2_CONFIG = {'hidden_size': 300, 'depth': 3, 'epochs': 50}

    def _write_csv(path, smiles_list, targets=None):
        with open(path, 'w', newline='') as f:
            writer = csv.writer(f)
            if targets is not None:
                writer.writerow(['smiles', 'target'])
                for smi, y in zip(smiles_list, targets):
                    writer.writerow([smi, float(y)])
            else:
                writer.writerow(['smiles'])
                for smi in smiles_list:
                    writer.writerow([smi])

    mpnn2_tuned_results = []

    for col in LARGE_ENDPOINTS:
        ep = EP_SHORT_MAP[col]
        _, _, y_train, y_test, smiles_train, smiles_test = splits[col]

        ep_dir    = os.path.join(MPNN2_TUNE_DIR, ep)
        train_csv = os.path.join(ep_dir, 'train.csv')
        test_csv  = os.path.join(ep_dir, 'test.csv')
        preds_csv = os.path.join(ep_dir, 'preds.csv')
        os.makedirs(ep_dir, exist_ok=True)
        _write_csv(train_csv, smiles_train, y_train)
        _write_csv(test_csv,  smiles_test)

        train_args = TrainArgs().parse_args([
            '--data_path',          train_csv,
            '--dataset_type',       'regression',
            '--save_dir',           ep_dir,
            '--features_generator', 'rdkit_2d_normalized',
            '--no_features_scaling',
            '--metric',             'mae',
            '--epochs',             str(MPNN2_CONFIG['epochs']),
            '--hidden_size',        str(MPNN2_CONFIG['hidden_size']),
            '--depth',              str(MPNN2_CONFIG['depth']),
            '--num_folds',          '1',
            '--quiet',
        ])
        cross_validate(args=train_args, train_func=run_training)

        pred_args = PredictArgs().parse_args([
            '--test_path',          test_csv,
            '--preds_path',         preds_csv,
            '--checkpoint_dir',     ep_dir,
            '--features_generator', 'rdkit_2d_normalized',
            '--no_features_scaling',
        ])
        raw_preds = make_predictions(args=pred_args)
        y_pred = np.array([p[0] for p in raw_preds])

        r2   = r2_score(y_test, y_pred)
        rmse = float(np.sqrt(mse_fn(y_test, y_pred)))
        mpnn2_tuned_results.append({'ep_short': ep, 'model': 'MPNN2 (tuned)', 'R2': r2, 'RMSE': rmse})
        print(f'{ep:6s}  R²={r2:+.3f}  RMSE={rmse:.3f}')

    # Save config for documentation
    save_params(MPNN2_CONFIG, os.path.join(TUNED_PARAMS_DIR, 'mpnn2_config.json'))
    mpnn2_tuned_df = pd.DataFrame(mpnn2_tuned_results)

### 3.3 Re-evaluate tuned models on full training set (R², RMSE)

Tuning cells (3.1/3.1b) fit on 80% of train (held out 20% as val). Here we refit LightGBM and RF on the **full** training set using saved params, and load MPNN2 predictions from saved `preds.csv`. Results persisted to CSV.

In [ ]:
# ── 3.3  Re-evaluate tuned models on full training set ──────────────────────
# Tuning cells fit on 80% of train; here we refit on full train for final performance.
lgbm_reeval, rf_reeval, mpnn2_reeval = [], [], []

for col in MODEL_ENDPOINTS:
    ep = EP_SHORT_MAP[col]
    X_train, X_test, y_train, y_test, smiles_train, smiles_test = splits_rdkit[col]

    # LightGBM — load best params, refit on full training set
    lgbm_params = load_params(os.path.join(TUNED_PARAMS_DIR, f'{ep}_lgbm.json'))
    lgbm_model = make_model('LightGBM', lgbm_params).fit(X_train, y_train)
    m = evaluate_model(lgbm_model, X_test, y_test)
    lgbm_reeval.append({'ep_short': ep, 'model': 'LightGBM (tuned)', **m})
    print(f'LightGBM  {ep:6s}  R²={m["R2"]:+.3f}  RMSE={m["RMSE"]:.3f}')

    # RandomForest — same pattern
    rf_params = load_params(os.path.join(TUNED_PARAMS_DIR, f'{ep}_rf.json'))
    rf_model = make_model('RandomForest', rf_params).fit(X_train, y_train)
    m = evaluate_model(rf_model, X_test, y_test)
    rf_reeval.append({'ep_short': ep, 'model': 'RandomForest (tuned)', **m})
    print(f'RF        {ep:6s}  R²={m["R2"]:+.3f}  RMSE={m["RMSE"]:.3f}')

    # MPNN2 — load predictions from saved preds.csv (no retraining needed)
    preds_csv = os.path.join('..', 'models', 'mpnn2_tuned', ep, 'preds.csv')
    if os.path.exists(preds_csv):
        y_pred_mpnn2 = pd.read_csv(preds_csv).iloc[:, 1].values
        m = evaluate_model(None, None, y_test, y_pred=y_pred_mpnn2)
        mpnn2_reeval.append({'ep_short': ep, 'model': 'MPNN2 (tuned)', **m})
        print(f'MPNN2     {ep:6s}  R²={m["R2"]:+.3f}  RMSE={m["RMSE"]:.3f}')

    print()

lgbm_tuned_df = pd.DataFrame(lgbm_reeval)
rf_tuned_df   = pd.DataFrame(rf_reeval)

lgbm_tuned_df.to_csv(os.path.join(TUNED_PARAMS_DIR, 'lgbm_tuned_metrics.csv'), index=False)
rf_tuned_df.to_csv(  os.path.join(TUNED_PARAMS_DIR, 'rf_tuned_metrics.csv'),   index=False)

if mpnn2_reeval:
    mpnn2_tuned_df = pd.DataFrame(mpnn2_reeval)
    mpnn2_tuned_df.to_csv(os.path.join(TUNED_PARAMS_DIR, 'mpnn2_tuned_metrics.csv'), index=False)

# ── Summary comparison including BayesianRidge (non-tuned reference) ─────────
br_baseline = results_df[
    (results_df['features'] == 'RDKit2D') &
    (results_df['model'] == 'BayesianRidge') &
    (results_df['ep_short'].isin(['HLM', 'MDR1', 'SOL', 'RLM']))
][['ep_short', 'model', 'R2', 'RMSE']].copy()

all_tuned = pd.concat([br_baseline, lgbm_tuned_df, rf_tuned_df] +
                      ([mpnn2_tuned_df] if mpnn2_reeval else []),
                      ignore_index=True)[['ep_short', 'model', 'R2', 'RMSE']]

col_order = ['BayesianRidge', 'LightGBM (tuned)', 'RandomForest (tuned)'] + \
            (['MPNN2 (tuned)'] if mpnn2_reeval else [])

print('\nR² — Tuned models vs BayesianRidge reference:')
display(all_tuned.pivot(index='ep_short', columns='model', values='R2')[col_order].round(4))
print('\nRMSE — Tuned models vs BayesianRidge reference:')
display(all_tuned.pivot(index='ep_short', columns='model', values='RMSE')[col_order].round(4))

MAE, P and CCC unecessary here, more valuable in the learning curve and noise injection section, overall mpnn2 is the best performer

# 

1. be clear on inputs and output for dataset, what models are taking in for input, what they are predicting
2. distribution of similarities of chemical compounds, did look at the endpoints, need to look at pairwise tanimoto similartiy distribution, how diverse chemical space is, function bulk tanimotos similarty - function is quick look it up, worth understanding how its done. 
3. interesrintg to look at morgan vs rdkit descriptors vs graph based method

section 3 can't run standalone. The minimum chain to get there is:                                                                            

  cell 1 (imports + constants)                                                                                                                                                 
    → 1.1 (load df)
      → 2.1 (featurize → builds splits_ecfp, splits_rdkit)
        → 2.2 (train/test split)
          → Part 3 ✓